In [ ]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import sys, gc

sys.path.append("..")
from Network.index import ModelNetwork
from utils.index import showTile
from Network.index import ModelNetwork
import torch, torchvision # pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
from sklearn.model_selection import train_test_split, GroupKFold, StratifiedKFold

from PIL import Image
import matplotlib.pyplot as plt
import cv2, os, shutil, glob, json, math, datetime, random, gc, ast
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from time import time
from utils.index import *
from tqdm import tqdm

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import nms

from torch.utils.data import DataLoader, Dataset
from torch.utils.data.sampler import SequentialSampler

import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

from utils.Plotter.index import Plotter
pd.set_option('display.max_columns', None)

In [2]:
def showTile(img=None, mask=None, save=None):
    if img is None and mask is None:
        return print("Erro: Forneça pelo menos 'img' ou 'mask'.")

    ref_vol = img if img is not None else mask
    mid_x = ref_vol.shape[0] // 2
    mid_y = ref_vol.shape[1] // 2
    mid_z = ref_vol.shape[2] // 2

    def get_slices(vol):
        if vol is None:
            return None
        
        s_x = np.array(vol[mid_x, :, :]) # Plano YZ
        s_y = np.array(vol[:, mid_y, :]) # Plano XZ
        s_z = np.array(vol[:, :, mid_z]) # Plano XY
        return [s_x, np.rot90(s_z, -1), s_y]

    img_slices  = get_slices(img)
    mask_slices = get_slices(mask)
    cmap_mask_only    = ListedColormap(['black', 'red', 'green', 'blue'])
    cmap_mask_overlay = ListedColormap([(0, 0, 0, 0), 'red', 'green', 'blue'])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles    = [f'Slice X={mid_x}', f'Slice Y={mid_y}', f'Slice Z={mid_z}']

    for i, ax in enumerate(axes):
        if img is not None:
            ax.imshow(img_slices[i], cmap='gray')
            
        if mask is not None:
            if img is not None:
                ax.imshow(mask_slices[i], cmap=cmap_mask_overlay, vmin=0, vmax=3, alpha=0.6)
            else:
                ax.imshow(mask_slices[i], cmap=cmap_mask_only, vmin=0, vmax=3)
        
        ax.set_title(titles[i])

    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)
        return plt.close(fig)

    plt.show()

In [3]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


In [4]:
#BASE_PATH = r'../Lucas'
BASE_PATH = r'Backup'

In [5]:
models  = [path for path in os.listdir(BASE_PATH)]
patches = ['patch_1200', 'patch_1300', 'patch_1400', 'patch_2600']
models, patches

(['model_1'], ['patch_1200', 'patch_1300', 'patch_1400', 'patch_2600'])

In [6]:
def getFiles(path, limit=None, shuffle=False):
    target = sorted([os.path.abspath(p) for p in glob.glob(os.path.join(path, '*'))])
    if shuffle:
        np.random.shuffle(target)
    return target[:limit]

def getDAT(path):
    return np.fromfile(path, dtype=np.float32).reshape((128, 128, 128))

class PredictDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row  = self.df.iloc[index]
        img  = getDAT(row.img_path)
        img_tensor  = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        return (img_tensor)

In [7]:
for model_name in models:
    model_path = f'{BASE_PATH}/{model_name}' 
    
    with open(f'{model_path}/info.json', 'r', encoding='utf-8') as f:
        modelInfo = json.load(f)

    modelOptions = modelInfo.get('model', {})
    print(f"\n[{model_name}] Model Options:")
    print(json.dumps(modelOptions, indent=4))

    network   = ModelNetwork(**modelOptions)
    modelData = torch.load(f'{model_path}/model.pth')

    network.model.load_state_dict(modelData['model'])
    network.model.eval()
    network.model.to(network.device)
    print(f"Weights from {model_path} loaded successfully!\n")

    for patch_id in patches:
        img_paths = getFiles(f'../Dataset/marlim/{patch_id}/images')
        df = pd.DataFrame({'img_path': img_paths, 'shape': '(128, 128, 128)'})
        
        predictDataset = PredictDataset(df)
        predictLoader  = DataLoader(
            predictDataset, 
            batch_size=1,
            shuffle=False, 
            num_workers=2,
            pin_memory=True if torch.cuda.is_available() else False
        )
        
        mask_dat_dir  = os.path.join(model_path, 'marlim', patch_id, 'masks')
        os.makedirs(mask_dat_dir, exist_ok=True)
        print(f"Iniciando predições para {model_name} - {patch_id}...")

        with torch.no_grad():
            for index, img_tensor in enumerate(tqdm(predictLoader)):
                orig_path = df.iloc[index]['img_path']
                base_name = os.path.basename(orig_path).replace('.dat', '')
                filename_dat = f"{base_name}.dat"

                img_batch = img_tensor.to(network.device)
                logits = network.model(img_batch)

                if hasattr(network, 'multiclass') and network.multiclass:
                    probs = torch.softmax(logits, dim=1)
                else:
                    probs = torch.sigmoid(logits)

                probs_np = probs.squeeze().cpu().numpy().astype(np.float32)
                probs_np.tofile(os.path.join(mask_dat_dir, filename_dat))

        print(f"Predições para {model_name} - {patch_id} salvas com sucesso!\n")


[model_1] Model Options:
{
    "network": "resaceunet",
    "img_size": [
        128,
        128,
        128
    ],
    "classes": 1,
    "channels": 1,
    "dropout": 0.1,
    "num_filters": 16,
    "lr": 0.0005
}
Weights from Backup/model_1 loaded successfully!

Iniciando predições para model_1 - patch_1200...


100%|██████████| 910/910 [01:52<00:00,  8.11it/s]


Predições para model_1 - patch_1200 salvas com sucesso!

Iniciando predições para model_1 - patch_1300...


100%|██████████| 910/910 [01:54<00:00,  7.96it/s]


Predições para model_1 - patch_1300 salvas com sucesso!

Iniciando predições para model_1 - patch_1400...


100%|██████████| 910/910 [01:55<00:00,  7.87it/s]


Predições para model_1 - patch_1400 salvas com sucesso!

Iniciando predições para model_1 - patch_2600...


100%|██████████| 910/910 [01:55<00:00,  7.85it/s]

Predições para model_1 - patch_2600 salvas com sucesso!

